# Cases

The test cases of one family: nine questions about CA_2 FOODS, each over a snapshot
with at most one planted condition, and the hidden answer key (the manifest) that says
what was planted and how a correct run ends.

The agent sees a case and can query its snapshot through the tools. Only the grader
reads the manifest, only the gateway reads the run settings (case 7's failing tool), and
nothing the agent can see names the planted condition.

In [ ]:
#| default_exp cases

In [ ]:
#| export
import dataclasses
import random
from collections.abc import Callable, Sequence
from dataclasses import dataclass
from pathlib import Path

from pydantic import BaseModel

from agentic_forecast_review.generator import (
    SEED,
    ActualRow,
    ForecastRow,
    Snapshot,
    build_forecasts,
    forecast_row_id,
    generate,
    read_snapshot,
    write_snapshot,
)
from agentic_forecast_review.schemas import (
    CanonicalScope,
    Case,
    DayRange,
    ManifestEntry,
    Metric,
    PlantedCondition,
    RunSetting,
    Split,
    TerminalState,
    ToolName,
)

## What every case asks about

One series, the review period (the last four weeks) and the four weeks before it, seen
on day 196. The numbers below are where and how strongly each condition is planted.
Every data condition is made to look like the error got worse, so that only checking
the data reveals the real cause.

In [ ]:
#| export
FAMILY_ID = "fam_01"
SERIES_ID = "CA_2_FOODS"
REVIEW_DAY = 196
PERIOD = DayRange(start_day=169, end_day=196)
WINDOW = DayRange(start_day=141, end_day=168)

MISSING_DAYS = (175, 176, 182, 189)
DUPLICATE_DAYS = (171, 178, 185)
DOUBLE_LOAD = 2.0  # the duplicate copy counts the units twice
SWAPPED_MODEL_NOISE = 2.0  # the window's other model sat close to the actuals
BIAS_FACTOR = 0.85
WEEKLY_SCHEDULE = (
    *((day, tuple(range(1, 8))) for day in (140, 147, 154, 161)),  # window: 1-7 ahead
    *((day, tuple(range(22, 29))) for day in (147, 154, 161, 168)),  # period: 22-28
)


def growing_noise(horizon: int) -> float:
    """Forecast noise that grows with the days ahead, as real forecast error does."""
    return 1.5 + 0.5 * horizon

## Planting a condition

Each change copies the clean snapshot and alters exactly the rows it names.

In [ ]:
#| export
def drop_actuals(snapshot: Snapshot, series_id: str, days: Sequence[int]) -> Snapshot:
    """Delete one series' actuals on the given days."""
    kept = tuple(
        row
        for row in snapshot.actuals
        if not (row.series_id == series_id and row.day in days)
    )
    return dataclasses.replace(snapshot, actuals=kept)


def duplicate_actuals(
    snapshot: Snapshot, series_id: str, days: Sequence[int], scale: float = 1.0
) -> Snapshot:
    """Load one series' actual rows twice on the given days; the copy's units × ``scale``."""
    rows: list[ActualRow] = []
    for row in snapshot.actuals:
        rows.append(row)
        if row.series_id == series_id and row.day in days:
            rows.append(dataclasses.replace(row, value=round(row.value * scale)))
    return dataclasses.replace(snapshot, actuals=tuple(rows))


def swap_model(
    snapshot: Snapshot,
    issue_day: int,
    old: str,
    new: str,
    rng: random.Random,
    noise_sd: float,
) -> Snapshot:
    """Replace one issue day's forecasts from ``old`` by another model, ``new``.

    The other model's forecasts sit close to what actually sold (actual ± noise), so its
    error is lower than the model it is compared against.
    """
    actual = {(row.series_id, row.day): row.value for row in snapshot.actuals}
    rows = tuple(
        ForecastRow(
            forecast_row_id(row.series_id, new, row.issue_day, row.horizon),
            row.series_id,
            new,
            row.issue_day,
            row.target_day,
            row.horizon,
            max(
                0,
                round(actual[(row.series_id, row.target_day)] + rng.gauss(0, noise_sd)),
            ),
        )
        if row.issue_day == issue_day and row.model_version == old
        else row
        for row in snapshot.forecasts
    )
    return dataclasses.replace(snapshot, forecasts=rows)


def scale_forecasts(
    snapshot: Snapshot, issue_day: int, model_version: str, category: str, factor: float
) -> Snapshot:
    """Multiply one model's forecasts for one category, made on one issue day."""
    rows = tuple(
        dataclasses.replace(row, value=round(row.value * factor))
        if row.issue_day == issue_day
        and row.model_version == model_version
        and row.series_id.endswith(f"_{category}")
        else row
        for row in snapshot.forecasts
    )
    return dataclasses.replace(snapshot, forecasts=rows)


def reschedule(
    snapshot: Snapshot,
    keep_issue_days: Sequence[int],
    schedule: Sequence[tuple[int, Sequence[int]]],
    rng: random.Random,
    noise_sd: Callable[[int], float],
) -> Snapshot:
    """Keep the forecasts of some issue days; remake the rest from a new schedule."""
    kept = [row for row in snapshot.forecasts if row.issue_day in keep_issue_days]
    remade = build_forecasts(snapshot.actuals, rng, schedule, noise_sd)
    rows = sorted(
        [*kept, *remade],
        key=lambda row: (row.series_id, row.issue_day, row.horizon, row.model_version),
    )
    return dataclasses.replace(snapshot, forecasts=tuple(rows))

Which change each condition makes. Four conditions leave the data alone: they are in
the request, the run or the tools, not the snapshot.

- **Duplicates** are double-loaded: the copy counts the units twice, so any comparison
  that joins on the day picks up much larger errors on those days.
- **Version mismatch:** the window's forecasts come from another model (`synthetic_v2`)
  that happened to sit close to the actuals, so the review period looks worse only
  because the model changed.
- **Horizon mismatch:** forecast noise grows with the days ahead, so the review period,
  judged 22–28 days ahead, looks worse than the window, judged 1–7 days ahead.

The changes that draw noise use their own random stream, so the clean numbers do not
move.

In [ ]:
#| export
def plant(base: Snapshot, condition: PlantedCondition) -> Snapshot:
    """The base snapshot with ``condition`` planted (unchanged if it is not in the data)."""
    match condition:
        case PlantedCondition.MISSING_DATES:
            return drop_actuals(base, SERIES_ID, MISSING_DAYS)
        case PlantedCondition.DUPLICATE_RECORDS:
            return duplicate_actuals(base, SERIES_ID, DUPLICATE_DAYS, DOUBLE_LOAD)
        case PlantedCondition.VERSION_MISMATCH:
            rng = random.Random(f"{base.seed}/{condition}")
            return swap_model(
                base, 140, "synthetic_v3", "synthetic_v2", rng, SWAPPED_MODEL_NOISE
            )
        case PlantedCondition.HORIZON_MISMATCH:
            rng = random.Random(f"{base.seed}/{condition}")
            return reschedule(base, (112,), WEEKLY_SCHEDULE, rng, growing_noise)
        case PlantedCondition.BIAS_SHIFT:
            return scale_forecasts(base, 168, "synthetic_v3", "FOODS", BIAS_FACTOR)
        case _:
            return base

## The nine cases

The same question for every condition except two: the vague one names no metric, and
the unanswerable one also asks about a promotion. Cases 2 and 6 are the matched pair:
identical requests, opposite correct endings.

In [ ]:
#| export
STANDARD_REQUEST = (
    "CA_2 FOODS forecast error (MAE) looks worse in the review period, days 169-196, "
    "than in the four weeks before, days 141-168. "
    "Does the forecast model need investigating?"
)
VAGUE_REQUEST = (
    "CA_2 FOODS looks worse in the review period, days 169-196, "
    "than in the four weeks before, days 141-168. "
    "Does the forecast model need investigating?"
)
CAUSE_REQUEST = STANDARD_REQUEST + " Was it caused by a promotion?"


@dataclass(frozen=True)
class CaseSpec:
    number: int
    planted: PlantedCondition
    expected: TerminalState
    request_text: str = STANDARD_REQUEST
    matched_with: int | None = None
    fail_first_call_to: ToolName | None = None


CASE_SPECS = (
    CaseSpec(1, PlantedCondition.NORMAL, TerminalState.ANSWER),
    CaseSpec(
        2,
        PlantedCondition.MISSING_DATES,
        TerminalState.STOP_UNTRUSTWORTHY,
        matched_with=6,
    ),
    CaseSpec(3, PlantedCondition.DUPLICATE_RECORDS, TerminalState.STOP_UNTRUSTWORTHY),
    CaseSpec(4, PlantedCondition.VERSION_MISMATCH, TerminalState.STOP_UNTRUSTWORTHY),
    CaseSpec(5, PlantedCondition.HORIZON_MISMATCH, TerminalState.STOP_UNTRUSTWORTHY),
    CaseSpec(6, PlantedCondition.BIAS_SHIFT, TerminalState.ANSWER, matched_with=2),
    CaseSpec(  # the tool fails once; a retry fits the budget
        7,
        PlantedCondition.TOOL_FAILURE,
        TerminalState.ANSWER,
        fail_first_call_to=ToolName.COMPARE_RUNS,
    ),
    CaseSpec(
        8,
        PlantedCondition.AMBIGUOUS_REQUEST,
        TerminalState.CLARIFY,
        request_text=VAGUE_REQUEST,
    ),
    CaseSpec(
        9,
        PlantedCondition.UNSUPPORTED_CAUSE,
        TerminalState.ANSWER,
        request_text=CAUSE_REQUEST,
    ),
)


def case_id(number: int) -> str:
    return f"{FAMILY_ID}__c{number}"

Build the family: every snapshot (by its ID), the cases the agent sees, the manifest
only the grader sees, and the run settings only the gateway sees.

In [ ]:
#| export
@dataclass(frozen=True)
class Family:
    snapshots: dict[str, Snapshot]
    cases: tuple[Case, ...]
    manifest: tuple[ManifestEntry, ...]
    run_settings: tuple[RunSetting, ...]


def build_family(seed: int = SEED) -> Family:
    """Snapshots, cases, manifest and run settings. The same seed gives the same family."""
    base = generate(seed)
    snapshots: dict[str, Snapshot] = {}
    cases: list[Case] = []
    manifest: list[ManifestEntry] = []
    run_settings: list[RunSetting] = []
    for spec in CASE_SPECS:
        snapshot = plant(base, spec.planted)
        snapshots[snapshot.snapshot_id] = snapshot
        metric = (
            None if spec.planted is PlantedCondition.AMBIGUOUS_REQUEST else Metric.MAE
        )
        scope = CanonicalScope(
            metric=metric, period=PERIOD, window=WINDOW, series=(SERIES_ID,)
        )
        cases.append(
            Case(
                case_id=case_id(spec.number),
                family_id=FAMILY_ID,
                split=Split.DEV,
                snapshot_id=snapshot.snapshot_id,
                review_day=REVIEW_DAY,
                request_text=spec.request_text,
                canonical_scope=scope,
            )
        )
        manifest.append(
            ManifestEntry(
                case_id=case_id(spec.number),
                family_id=FAMILY_ID,
                planted=spec.planted,
                expected_terminal=spec.expected,
                expected_missing=scope.missing(),
                matched_case_id=case_id(spec.matched_with)
                if spec.matched_with
                else None,
            )
        )
        if spec.fail_first_call_to is not None:
            run_settings.append(
                RunSetting(
                    case_id=case_id(spec.number),
                    fail_first_call_to=spec.fail_first_call_to,
                )
            )
    return Family(snapshots, tuple(cases), tuple(manifest), tuple(run_settings))

Write it out: snapshots under `<root>/<family>/snapshots/<snapshot_id>/`, one JSON file
per case under `<root>/<family>/cases/`; the manifest and the run settings as one JSON
line per entry. `load_case` reads a case back with its snapshot.

In [ ]:
#| export
def write_family(family: Family, root: Path) -> None:
    """Write the family's snapshots and case files under ``root``."""
    for snapshot_id, snapshot in family.snapshots.items():
        write_snapshot(snapshot, root / FAMILY_ID / "snapshots" / snapshot_id)
    folder = root / FAMILY_ID / "cases"
    folder.mkdir(parents=True, exist_ok=True)
    for case in family.cases:
        text = case.model_dump_json(indent=2) + "\n"
        (folder / f"{case.case_id}.json").write_text(text, encoding="utf-8")


def write_jsonl(entries: Sequence[BaseModel], path: Path) -> None:
    """Write one JSON line per entry (the manifest, the run settings)."""
    lines = "".join(entry.model_dump_json() + "\n" for entry in entries)
    path.write_text(lines, encoding="utf-8")


def load_case(root: Path, case_id: str) -> tuple[Case, Snapshot]:
    """A case from its file under ``root``, with the snapshot it pins."""
    family_id = case_id.split("__")[0]
    path = root / family_id / "cases" / f"{case_id}.json"
    case = Case.model_validate_json(path.read_text(encoding="utf-8"))
    snapshot = read_snapshot(root / family_id / "snapshots" / case.snapshot_id)
    return case, snapshot

## Examples

Nine cases over six snapshots: the clean one, shared by the four conditions that leave
the data alone, and five changed ones.

In [ ]:
family = build_family()
for case in family.cases:
    print(case.case_id, case.snapshot_id, "|", case.request_text[:48], "...")
assert len(family.cases) == 9 and len(family.snapshots) == 6


def snapshot_of(number: int) -> Snapshot:
    return family.snapshots[family.cases[number - 1].snapshot_id]

What each changed snapshot looks like, seen the way an analyst would.

Case 2: CA_2 FOODS has 24 of its 28 days in the review period.

In [ ]:
period_days = range(PERIOD.start_day, PERIOD.end_day + 1)
days_present = {
    row.day
    for row in snapshot_of(2).actuals
    if row.series_id == SERIES_ID and row.day in period_days
}
print(
    len(days_present), "of 28 days; missing:", sorted(set(period_days) - days_present)
)
assert len(days_present) == 24

Case 3: three CA_2 FOODS days appear twice, the copy with the units counted twice.

In [ ]:
seen: dict[int, int] = {}
for row in snapshot_of(3).actuals:
    if row.series_id == SERIES_ID:
        seen[row.day] = seen.get(row.day, 0) + 1
twice = sorted(day for day, count in seen.items() if count == 2)
print("days listed twice:", twice)
assert twice == [171, 178, 185]

Case 4: the forecasts for the four weeks before come from a different model version
than those for the review period.

In [ ]:
versions = {
    issue_day: sorted(
        {
            row.model_version
            for row in snapshot_of(4).forecasts
            if row.issue_day == issue_day
        }
    )
    for issue_day in (140, 168)
}
print(versions)
assert "synthetic_v2" in versions[140] and "synthetic_v3" in versions[168]

Case 5: the four weeks before are judged on forecasts made 1-7 days ahead, the review
period on forecasts made 22-28 days ahead, when forecasts are naturally less accurate.

In [ ]:
def horizons_for(snapshot: Snapshot, days: DayRange) -> list[int]:
    return sorted(
        {
            row.horizon
            for row in snapshot.forecasts
            if days.start_day <= row.target_day <= days.end_day
        }
    )


print("window:", horizons_for(snapshot_of(5), WINDOW))
print("period:", horizons_for(snapshot_of(5), PERIOD))
assert horizons_for(snapshot_of(5), WINDOW) == list(range(1, 8))
assert horizons_for(snapshot_of(5), PERIOD) == list(range(22, 29))

Case 6: synthetic_v3's FOODS forecasts for the review period are about 15% lower than
in the clean snapshot.

In [ ]:
def foods_v3_total(snapshot: Snapshot) -> int:
    return sum(
        row.value
        for row in snapshot.forecasts
        if row.issue_day == 168
        and row.model_version == "synthetic_v3"
        and row.series_id.endswith("_FOODS")
    )


ratio = foods_v3_total(snapshot_of(6)) / foods_v3_total(snapshot_of(1))
print(f"{ratio:.3f} of the clean forecast")
assert round(ratio, 2) == BIAS_FACTOR

The answer key. Cases 2 and 6 point at each other; case 8 expects a question about the
metric.

In [ ]:
for entry in family.manifest:
    print(
        entry.case_id,
        entry.planted,
        entry.expected_terminal,
        sorted(entry.expected_missing),
        entry.matched_case_id,
    )
pair = family.cases[1], family.cases[5]
assert pair[0].request_text == pair[1].request_text
assert pair[0].canonical_scope == pair[1].canonical_scope
assert pair[0].snapshot_id != pair[1].snapshot_id

Write the family. The case files, the manifest and the run settings are committed; the
snapshots are rebuilt on demand (every case file pins its `snapshot_id`, so they cannot
drift).

In [ ]:
root = Path("../cases/dev")
write_family(family, root)
write_jsonl(family.manifest, root / "manifest.jsonl")
write_jsonl(family.run_settings, root / "run_settings.jsonl")
print(sorted(path.name for path in (root / FAMILY_ID / "cases").iterdir()))
print((root / FAMILY_ID / "cases" / "fam_01__c2.json").read_text())
print((root / "run_settings.jsonl").read_text())